# A Multi Index RAG Pipeline

Both Semantic and Lexical Search share similar implementations.
- `add_document()`
- `search()`

Going to wrap these into a `Retriever` class to merge the results together

## Reciprocal Rank Fusion 
Combining the outputs based on the reciprocal ranks of each chunk

<img src="./supporting_materials/Screenshot 2026-07-20 at 1.17.53 pm.png">

Final rank would be Section 2, 6, then 7

In [1]:
from utils.retriever import Retriever
from utils.bm25 import BM25Index
from utils.vector_index import VectorIndex
from utils.search_index import SearchIndex
from utils.util_funcs import *

Initialised <anthropic.Anthropic object at 0x114aac440> with the model: claude-haiku-4-5


/Users/aidanlockwood/Library/CloudStorage/OneDrive-BluPanteraTechPtyLtd/GitHub/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Chunking the Source Text

In [2]:
with open("./supporting_materials/report.md", "r") as f:
    text = f.read()

chunks = chunk_by_section(text)

chunks[3]

'Methodology\n\nThe insights compiled within this Annual Interdisciplinary Research Review represent a synthesis of findings drawn from standard departmental reporting cycles, specialized project updates, and cross-functional review meetings conducted throughout the year. Data sources included internal project databases, laboratory notebooks, financial reporting systems, legal case summaries, security incident logs, and minutes from dedicated working groups. A central review committee, comprising representatives nominated by each division head, was tasked with identifying key developments and potential cross-domain implications. This committee utilized a standardized reporting template to capture essential details, including unique identifiers (project codes, error numbers, case references, etc.) and progress metrics. Subsequent analysis focused on identifying thematic overlaps, shared challenges, and opportunities for synergistic development, forming the basis of this consolidated rep

## Step 2: Generate Embeddings

In [3]:
embeddings = generate_embedding(chunks, input_type="document", cache_path="./supporting_materials/embeddings.json")

## Step 3: Storing Embeddings into Vector Database

In [4]:
store = VectorIndex()

for embedding, chunk in zip(embeddings, chunks):
    store.add_vector(embedding, { 
        'content': chunk
    })

## Step 4: Embedding User Query
User will ask a question. Generate an embedding for it

In [5]:
user_embedding = generate_embedding("What happened with INC-2023-Q4-011?")

In [6]:
results = store.search(user_embedding, 2)

for doc, distance in results: 
    print(distance, '\n', doc['content'][0:200], '\n')

0.571287093367925 
 Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011

The Cybersecurity Operations Center successfully contained and remediated a targeted intrusion attempt tracked as `INC-2 

0.5769424057153363 
 Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems 



## Testing the Retriever Class

In [7]:
vector_index = VectorIndex(embedding_fn = generate_embedding)
bm25_index = BM25Index()

retriever = Retriever(bm25_index, vector_index)

## Loading in the JSON file of the Chunks from a Previous Notebook

In [8]:
from pathlib import Path
import json

chunks_path = Path("./supporting_materials/chunks.json")

if chunks_path.exists():
    with chunks_path.open("r", encoding="utf-8") as f:
        chunks = json.load(f)
else:
    with open("./supporting_materials/report.md", "r") as f:
        text = f.read()

    chunks = chunk_by_section(text)

    with chunks_path.open("w", encoding="utf-8") as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)

chunks_path


PosixPath('supporting_materials/chunks.json')

In [9]:
retriever.add_documents([
    {'content': chunk} for chunk in chunks
])

In [11]:
results = retriever.search('What happened with INC-2023-Q4-011?', 3)

for doc, score in results:
    print(score, '\n', doc['content'][0:200], '\n---\n')

0.03252247488101534 
 Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems 
---

0.03252247488101534 
 Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011

The Cybersecurity Operations Center successfully contained and remediated a targeted intrusion attempt tracked as `INC-2 
---

0.031024531024531024 
 Methodology

The insights compiled within this Annual Interdisciplinary Research Review represent a synthesis of findings drawn from standard departmental reporting cycles, specialized project updates 
---

